# Kaggle Playground S6E3 — Customer Churn: An Educational Walkthrough

This notebook is a heavily commented version of a competitive Kaggle solution.
The goal is not just to get a good score, but to explain **why** each technique works.

## What we cover
1. **Feature Engineering** — creating signals the model can't find on its own
2. **Out-of-Fold (OOF) Predictions** — honest model evaluation without data leakage
3. **Target Encoding** — turning categories into numbers using the target variable
4. **Multiple Seeds** — reducing variance through randomness averaging
5. **Ensemble + Hill Climbing** — combining diverse models optimally
6. **Pseudo-Labeling** — using unlabelled test data to improve training

## The Problem
Predict which telecom customers will cancel their subscription (churn).
Scored on **AUC-ROC**: we're ranking customers by churn probability, not just classifying them.
Top leaderboard score: ~0.9176. We're aiming to get close.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

SEED = 42      # Master random seed — fix this so results are reproducible
N_SPLITS = 5   # Number of cross-validation folds
np.random.seed(SEED)
print('Libraries loaded')

## 1. Load Data

In [ ]:
# Kaggle kernels store data at /kaggle/input/. Locally we use a data/ folder.
# The os.path.exists check lets the same notebook run in both environments.
if os.path.exists('/kaggle/input'):
    TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e3/train.csv'
    TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e3/test.csv'
    SUB_PATH   = '/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv'
    ORIG_PATH  = '/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv'
else:
    TRAIN_PATH = 'data/train.csv'
    TEST_PATH  = 'data/test.csv'
    SUB_PATH   = 'data/sample_submission.csv'
    ORIG_PATH  = 'data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sub   = pd.read_csv(SUB_PATH)

# The original IBM Telco dataset (7,043 real customers) that this synthetic
# competition was generated from. We'll use it to extract ground-truth statistics.
orig  = pd.read_csv(ORIG_PATH)

print('Train:', train.shape, '| Test:', test.shape, '| Orig:', orig.shape)

## 2. Data Preparation

In [ ]:
TARGET = 'Churn'

# These are the categorical columns — they contain string values like 'Yes'/'No',
# 'Month-to-month', 'Fiber optic' etc.
CATS = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod',
]

# Save IDs before dropping — we'll need them for the submission file
train_ids = train['id'].copy()
test_ids  = test['id'].copy()
train = train.drop(columns=['id'])
test  = test.drop(columns=['id'])
if 'customerID' in orig.columns:
    orig = orig.drop(columns=['customerID'])

# Convert string target 'Yes'/'No' to binary 1/0
def map_target(s):
    return s.map({'Yes': 1, 'No': 0}).astype('int8') if s.dtype == object else s.astype('int8')

train[TARGET] = map_target(train[TARGET])
orig[TARGET]  = map_target(orig[TARGET])

# Clean numeric columns. TotalCharges has some blank strings (customers with
# tenure=0 have no charges yet), so we fill those with tenure * MonthlyCharges.
for df in [train, test, orig]:
    df['tenure']         = pd.to_numeric(df['tenure'], errors='coerce').astype('float32')
    df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce').astype('float32')
    df['TotalCharges']   = pd.to_numeric(
        df['TotalCharges'].astype(str).str.strip().replace('', np.nan),
        errors='coerce'
    ).astype('float32')
    mask = df['TotalCharges'].isna()
    df.loc[mask, 'TotalCharges'] = df.loc[mask, 'tenure'] * df.loc[mask, 'MonthlyCharges']
    # Ensure all categoricals are strings so we can concatenate them later
    for c in CATS:
        if c in df.columns:
            df[c] = df[c].astype(str).fillna('Missing')

BASE = [c for c in train.columns if c != TARGET]
print('BASE cols:', len(BASE), '| Churn rate:', round(train[TARGET].mean(), 4))

## 3. Feature Engineering

Tree models like LightGBM are powerful, but they can only split on one feature at a time.
If the signal lives in a *combination* of features (e.g. fiber optic + month-to-month contract
= very high churn), the model needs many splits to find it.

We help the model by creating features explicitly. Four types below.

In [ ]:
# ── DIGIT FEATURES ────────────────────────────────────────────────────────────
#
# WHY: Telecom pricing clusters at round numbers ($29.99, $49.99, $79.99).
# A tree needs a lucky split threshold to find these. By decomposing
# MonthlyCharges=7450 (×100) into digits [7,4,5,0], we make the leading digit
# directly available as a feature. The model can now split on 'first digit = 7'
# in one step instead of needing exact thresholds.
#
def add_digit_cols(df, col, multiplier):
    if multiplier == 'direct':
        vals = pd.to_numeric(df[col], errors='coerce').fillna(-1).astype(int)
    else:
        vals = (pd.to_numeric(df[col], errors='coerce') * multiplier).round(0).fillna(-1).astype(int)
    max_len = {'tenure': 3, 'MonthlyCharges': 5, 'TotalCharges': 6}.get(col, vals.astype(str).str.len().max())
    s = vals.astype(str).str.zfill(max_len)  # zero-pad so digit positions are consistent
    created = []
    for i in range(max_len):
        n = f'{col}_DIGIT_{i+1}'
        df[n] = s.str[i].astype(int)
        created.append(n)
    return created

DIGIT = []
for df in [train, test]:
    d = add_digit_cols(df, 'tenure', 'direct')         # tenure 0-72 → 3 digits
    add_digit_cols(df, 'MonthlyCharges', 100)           # e.g. 74.50 → 7450 → 5 digits
    add_digit_cols(df, 'TotalCharges', 1)               # integer part → 6 digits
    if df is train:
        DIGIT = d + [f'MonthlyCharges_DIGIT_{i+1}' for i in range(5)] + \
                    [f'TotalCharges_DIGIT_{i+1}' for i in range(6)]
print(len(DIGIT), 'DIGIT features')


# ── ROUND FEATURES ────────────────────────────────────────────────────────────
#
# WHY: Similar idea to digits. Customers on $79.99/month vs $80.01/month are
# essentially the same, but a tree might split them differently. Rounding at
# multiple scales (to nearest $1, $10, $100, $1000) creates explicit groupings.
#
ROUND = []
for col, levels in [
    ('MonthlyCharges', [('1s', 0), ('10s', -1), ('100s', -2), ('1000s', -3)]),
    ('TotalCharges',   [('1s', 0), ('10s', -1), ('100s', -2), ('1000s', -3)]),
    ('tenure',         [('1s', 0), ('10s', -1)]),
]:
    for suffix, level in levels:
        n = f'{col}_ROUND_{suffix}'
        ROUND.append(n)
        for df in [train, test]:
            df[n] = df[col].round(level).fillna(-1).astype(int)
print(len(ROUND), 'ROUND features')


# ── ORIGINAL DATASET STATISTICS ───────────────────────────────────────────────
#
# WHY: This synthetic competition was generated from a real 7,043-row IBM Telco
# dataset. That real data contains the true churn rates per group. By computing
# 'what fraction of month-to-month customers churned in the original data?' and
# handing that number to the model, we're giving it ground-truth signal that the
# synthetic data approximates but doesn't perfectly capture.
#
ORIG_FEATS = []
orig_mean = float(orig[TARGET].mean())
for col in BASE:
    mean_map  = orig.groupby(col)[TARGET].mean()  # churn rate per category value
    count_map = orig.groupby(col).size()           # how many real customers had this value
    for df in [train, test]:
        df[f'orig_mean_{col}']  = df[col].map(mean_map).fillna(orig_mean)
        df[f'orig_count_{col}'] = df[col].map(count_map).fillna(0)
    ORIG_FEATS += [f'orig_mean_{col}', f'orig_count_{col}']
print(len(ORIG_FEATS), 'ORIG features')


# ── PAIRWISE INTERACTION FEATURES ─────────────────────────────────────────────
#
# WHY: A customer on a 'Month-to-month' contract with 'Fiber optic' internet
# churns much more than a 'Two year' contract + 'Fiber optic' customer.
# The individual values are the same, but the combination is the signal.
# We create a new string 'Month-to-month_Fiber optic' to capture this.
#
# We then target-encode these (see next cell) to convert them to numbers.
# 19 base features → 19×18/2 = 171 pairwise interactions.
#
INTER = []
for col1, col2 in combinations(BASE, 2):
    n = f'{col1}_{col2}'
    INTER.append(n)
    for df in [train, test]:
        df[n] = df[col1].astype(str) + '_' + df[col2].astype(str)
print(len(INTER), 'INTER features')

FEATURES = list(dict.fromkeys(BASE + ORIG_FEATS + INTER + ROUND + DIGIT))
print(len(FEATURES), 'total features')

## 4. Target Encoding + Out-of-Fold Predictions

### What is Target Encoding?
Target encoding replaces a categorical value with the mean of the target for that category.
For example, 'Month-to-month' might become 0.43 (43% churn rate) and 'Two year' becomes 0.03.

This turns high-cardinality string features into meaningful numbers the model can use directly.
The 171 interaction features (like 'Month-to-month_Fiber optic') are perfect candidates —
there are too many unique values to one-hot encode efficiently.

### The Leakage Problem
If you compute 'average churn for Month-to-month customers' using ALL training data, then
use that average as a feature to train on the same data, you've created **data leakage**:
the feature for each row already 'knows' whether that row churned (it was used to compute the average).
The model sees artificially informative features and overfits.

### The Fix: Out-of-Fold Encoding
For each row, compute the target encoding using ONLY data that row was NOT part of.
This is done by splitting training data into k folds:
- For fold 1's rows: compute encoding using folds 2,3,4,5
- For fold 2's rows: compute encoding using folds 1,3,4,5
- etc.

This ensures no row's encoding was computed with knowledge of its own target value.

In [ ]:
print('Precomputing target encoding for 171 interaction features...')
print('This takes ~20 minutes on CPU — the TE is done ONCE and reused by all models.')
t0 = time.time()

X_all     = train[FEATURES].copy()
y_all     = train[TARGET].copy()
Xtest_raw = test[FEATURES].copy()
gm        = float(y_all.mean())  # global mean — fallback for unseen categories

# OOF target encoding for training data
# For each fold, compute TE from the OTHER folds, apply to this fold.
# Result: every row's TE was computed without seeing its own target value.
oof_te = pd.DataFrame(index=X_all.index)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_i, (ti, vi) in enumerate(kf.split(X_all), 1):
    Xt = X_all.iloc[ti]   # training portion of this fold
    yt = y_all.iloc[ti]
    Xv = X_all.iloc[vi]   # validation portion — this is what we're encoding
    fm = float(yt.mean())
    for col in INTER:
        # Compute mean churn rate per interaction value using ONLY training portion
        mp = Xt.assign(_y=yt.values).groupby(col)['_y'].mean()
        # Apply to validation portion — no leakage!
        oof_te.loc[Xv.index, f'TE_{col}'] = Xv[col].map(mp).fillna(fm).values
    print(f'  TE fold {fold_i}/5 done — {time.time()-t0:.0f}s elapsed')

# For test data: use ALL training data to fit the encoding (no leakage concern here,
# test targets are unknown)
print('Computing test TE from full training data...')
full_map = {
    col: X_all.assign(_y=y_all.values).groupby(col)['_y'].mean()
    for col in INTER
}
test_te = pd.DataFrame(
    {f'TE_{col}': Xtest_raw[col].map(full_map[col]).fillna(gm) for col in INTER},
    index=Xtest_raw.index
)

# Drop the raw string interaction columns, replace with their numeric TE versions
X_all     = X_all.drop(columns=INTER)
Xtest_raw = Xtest_raw.drop(columns=INTER)
for col in INTER:
    X_all[f'TE_{col}']      = oof_te[f'TE_{col}']
    Xtest_raw[f'TE_{col}']  = test_te[f'TE_{col}']

print(f'TE done in {time.time()-t0:.0f}s. Final feature count: {X_all.shape[1]}')

## 5. LightGBM with Multiple Seeds

### Why Cross-Validation?
We want to know how well our model generalises to unseen data. If we just train on all data
and evaluate on the same data, the model looks better than it really is (it memorised the data).

With k-fold CV, we split training data into 5 parts:
- Train on folds 1,2,3,4 → evaluate on fold 5
- Train on folds 1,2,3,5 → evaluate on fold 4
- ... etc.

Each row gets a prediction from a model that NEVER saw it during training.
These are called **Out-of-Fold (OOF) predictions** — the gold standard for CV in Kaggle.

### Why Multiple Seeds?
LightGBM uses randomness (random feature subsets, random row subsets per tree).
Different random seeds produce slightly different models with slightly different errors.
Averaging 3 seeds is like running 3 independent experiments and averaging — the random
errors cancel out, leaving only the true signal. Easy 0.001-0.002 AUC gain for free.

In [ ]:
def factorize3(tr, val, te):
    """Convert string categoricals to integer codes consistently across train/val/test.
    We must factorize all three together so the same string gets the same integer code.
    If we factorized separately, 'Yes' might be 0 in train but 1 in test."""
    codes, _ = pd.factorize(pd.concat([tr, val, te], ignore_index=True).astype(str))
    n1, n2 = len(tr), len(val)
    return (
        pd.Series(codes[:n1], index=tr.index, dtype='int32'),
        pd.Series(codes[n1:n1+n2], index=val.index, dtype='int32'),
        pd.Series(codes[n1+n2:], index=te.index, dtype='int32'),
    )

LGB_SEEDS = [42, 123, 456]  # 3 different random initialisations

lgb_params = {
    'objective': 'binary',       # binary classification
    'metric': 'auc',             # optimise AUC during training
    'n_estimators': 2000,        # max trees — early stopping will use fewer
    'learning_rate': 0.05,       # small learning rate → more trees needed but better generalisation
    'num_leaves': 127,           # tree complexity (2^7-1) — larger = more complex = more overfit risk
    'subsample': 0.8,            # use 80% of rows per tree — reduces overfitting
    'colsample_bytree': 0.6,     # use 60% of features per tree — forces diversity
    'min_child_samples': 20,     # minimum rows per leaf — prevents tiny overfitted leaves
    'reg_alpha': 0.1,            # L1 regularisation
    'reg_lambda': 1.0,           # L2 regularisation
    'n_jobs': -1,
    'verbose': -1,
}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
# StratifiedKFold ensures each fold has the same churn rate as the full dataset
# (important when classes are imbalanced — 22.5% churn here)

# We accumulate OOF and test predictions across seeds, then divide to get the average
oof   = np.zeros(len(X_all),     dtype=np.float64)
tpred = np.zeros(len(Xtest_raw), dtype=np.float64)

for seed in LGB_SEEDS:
    print(f'\n=== LightGBM seed={seed} ===')
    oof_s = np.zeros(len(X_all),     dtype=np.float64)
    tp_s  = np.zeros(len(Xtest_raw), dtype=np.float64)

    for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
        fold_start = time.time()
        Xtr  = X_all.iloc[tri].copy();  Xval = X_all.iloc[vali].copy()
        ytr  = y_all.iloc[tri].copy();  yval = y_all.iloc[vali].copy()
        Xte  = Xtest_raw.copy()

        # LightGBM needs integer-coded categoricals (not strings)
        for c in CATS:
            if c in Xtr.columns:
                Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
        for c in Xtr.select_dtypes('object').columns:
            Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])

        model = lgb.LGBMClassifier(**{**lgb_params, 'random_state': seed})
        model.fit(
            Xtr, ytr,
            eval_set=[(Xval, yval)],
            # early_stopping: stop if AUC doesn't improve for 100 rounds
            # This prevents overfitting and saves time
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(False)]
        )

        # predict_proba returns [P(class=0), P(class=1)] — we want P(churn)
        oof_s[vali] = model.predict_proba(Xval)[:, 1]
        tp_s += model.predict_proba(Xte)[:, 1] / N_SPLITS

        print(f'  Fold {fold} AUC: {roc_auc_score(yval, oof_s[vali]):.5f}  '
              f'iter={model.best_iteration_}  time={time.time()-fold_start:.0f}s')

    print(f'  Seed {seed} OOF AUC: {roc_auc_score(y_all, oof_s):.5f}')
    oof   += oof_s / len(LGB_SEEDS)  # accumulate, divide later
    tpred += tp_s  / len(LGB_SEEDS)

print(f'\nLightGBM multi-seed OOF AUC: {roc_auc_score(y_all, oof):.5f}')
np.save('oof_lgb.npy', oof)
np.save('pred_lgb.npy', tpred)

## 6. CatBoost

### Why a second model?
Two models that make *identical* predictions gain nothing from blending.
The value comes from **diversity** — models that make *different* errors.

CatBoost is genuinely different from LightGBM despite both being gradient boosting:
- LightGBM: grows trees leaf-wise (finds the single best split anywhere in the tree)
- CatBoost: grows trees level-wise (symmetric, all splits at same depth level first)
- CatBoost has its own internal categorical encoding (ordered target statistics) that
  produces different representations of the same categories

These differences mean their errors are partially uncorrelated — blending them helps.

In [ ]:
# CatBoost can handle string categoricals natively — no need to factorize them.
# We tell it which columns are categorical via cat_features indices.
cat_cols = [c for c in CATS if c in X_all.columns]
cat_idx  = [X_all.columns.tolist().index(c) for c in cat_cols]

cb_params = dict(
    iterations=2000,
    learning_rate=0.05,
    depth=6,              # CatBoost uses symmetric trees — depth 6 = 64 leaves
    l2_leaf_reg=3,        # L2 regularisation
    random_seed=SEED,
    eval_metric='AUC',
    od_type='Iter',       # overfitting detector type
    od_wait=100,          # stop if no improvement for 100 iterations
    verbose=False,
    thread_count=-1,
    # task_type='GPU',    # uncomment if GPU available — ~10x faster
)

oof_cat  = np.zeros(len(X_all),     dtype=np.float32)
pred_cat = np.zeros(len(Xtest_raw), dtype=np.float32)
cv_start = time.time()

for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
    fold_start = time.time()
    Xtr  = X_all.iloc[tri].copy();  Xval = X_all.iloc[vali].copy()
    ytr  = y_all.iloc[tri].copy();  yval = y_all.iloc[vali].copy()
    Xte  = Xtest_raw.copy()

    model_cb = CatBoostClassifier(**cb_params)
    model_cb.fit(Xtr, ytr, cat_features=cat_idx,
                 eval_set=(Xval, yval), use_best_model=True)

    oof_cat[vali] = model_cb.predict_proba(Xval)[:, 1]
    pred_cat += model_cb.predict_proba(Xte)[:, 1] / N_SPLITS

    print(f'CatBoost Fold {fold} AUC: {roc_auc_score(yval, oof_cat[vali]):.5f}  '
          f'best_iter={model_cb.best_iteration_}  time={time.time()-fold_start:.0f}s')

print(f'\nCatBoost OOF AUC: {roc_auc_score(y_all, oof_cat):.5f}  total={time.time()-cv_start:.0f}s')
np.save('oof_cat.npy', oof_cat)
np.save('pred_cat.npy', pred_cat)

## 7. Neural Network

### Why include a neural net?
Tree models (LGB, CatBoost) make predictions by stepping through a series of if/else rules.
They're great at capturing sharp boundaries ('if tenure < 12 AND contract = month-to-month...').

Neural networks learn smooth, continuous functions instead. They excel at:
- Capturing gradual relationships (tenure effect gradually decreasing)
- Learning complex feature interactions through hidden layers

The errors they make are structurally different from tree errors, making them valuable in an ensemble
even if their solo AUC is slightly lower.

In [ ]:
def prep_for_nn(Xtr, Xval, Xte, cat_cols):
    """Neural networks need all-numeric inputs and benefit from feature scaling.
    1. Encode categoricals as integers (crude but functional for sklearn MLP)
    2. StandardScaler: subtract mean, divide by std — puts all features on same scale
       so no feature dominates just because it has larger values."""
    Xtr = Xtr.copy(); Xval = Xval.copy(); Xte = Xte.copy()
    for c in cat_cols:
        codes, _ = pd.factorize(
            pd.concat([Xtr[c], Xval[c], Xte[c]], ignore_index=True).astype(str)
        )
        n1, n2 = len(Xtr), len(Xval)
        Xtr[c]  = codes[:n1].astype('int32')
        Xval[c] = codes[n1:n1+n2].astype('int32')
        Xte[c]  = codes[n1+n2:].astype('int32')
    scaler = StandardScaler()
    Xtr  = pd.DataFrame(scaler.fit_transform(Xtr),  columns=Xtr.columns)
    Xval = pd.DataFrame(scaler.transform(Xval),     columns=Xval.columns)
    Xte  = pd.DataFrame(scaler.transform(Xte),      columns=Xte.columns)
    return Xtr, Xval, Xte

nn_params = dict(
    hidden_layer_sizes=(256, 128, 64),  # 3 hidden layers: 256 → 128 → 64 neurons
    activation='relu',                  # ReLU: simple, effective, avoids vanishing gradients
    solver='adam',                      # adaptive learning rate optimiser
    max_iter=200,
    early_stopping=True,                # use 10% of train data as internal validation
    validation_fraction=0.1,
    n_iter_no_change=15,                # stop if no improvement for 15 epochs
    random_state=SEED,
    verbose=False,
)

oof_nn  = np.zeros(len(X_all),     dtype=np.float32)
pred_nn = np.zeros(len(Xtest_raw), dtype=np.float32)
cv_start = time.time()

for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
    fold_start = time.time()
    Xtr  = X_all.iloc[tri].copy();  Xval = X_all.iloc[vali].copy()
    ytr  = y_all.iloc[tri].copy();  yval = y_all.iloc[vali].copy()
    Xte  = Xtest_raw.copy()

    Xtr, Xval, Xte = prep_for_nn(Xtr, Xval, Xte, cat_cols)

    model_nn = MLPClassifier(**nn_params)
    model_nn.fit(Xtr, ytr)

    oof_nn[vali] = model_nn.predict_proba(Xval)[:, 1]
    pred_nn += model_nn.predict_proba(Xte)[:, 1] / N_SPLITS

    print(f'NN Fold {fold} AUC: {roc_auc_score(yval, oof_nn[vali]):.5f}  '
          f'iters={model_nn.n_iter_}  time={time.time()-fold_start:.0f}s')

print(f'\nNN OOF AUC: {roc_auc_score(y_all, oof_nn):.5f}  total={time.time()-cv_start:.0f}s')
np.save('oof_nn.npy', oof_nn)
np.save('pred_nn.npy', pred_nn)

## 8. Hill Climbing Ensemble

### Why not just average all models equally?
Simple averaging works, but some models contribute more than others and the optimal
blend is rarely equal weights. Hill climbing finds the optimal weights automatically.

### How Hill Climbing works
1. Start with the single best model as the current blend
2. Try adding each other model at many different weights
3. Keep the combination that most improves OOF AUC
4. Repeat from step 2 with the new blend
5. Stop when no combination improves things

This is **greedy** — it finds a locally optimal solution, not necessarily globally optimal.
But in practice it works very well and is much faster than exhaustive search.

### Why use OOF predictions for hill climbing?
We use OOF predictions (not test predictions) to find the weights, because we know the
true labels for training data. The weights that maximise OOF AUC should generalise
to the test set. If we used test predictions we'd be optimising for the hidden test labels
which we don't have access to (and which would be cheating).

In [ ]:
models = {
    'lgb': (oof,     tpred),
    'cat': (oof_cat, pred_cat),
    'nn':  (oof_nn,  pred_nn),
}

print('Single model OOF AUCs:')
for name, (o, _) in models.items():
    print(f'  {name}: {roc_auc_score(y_all, o):.5f}')

# Start from the best single model
best_name = max(models, key=lambda n: roc_auc_score(y_all, models[n][0]))
best_auc  = roc_auc_score(y_all, models[best_name][0])
print(f'\nStarting from: {best_name} (AUC={best_auc:.5f})')

weights    = {best_name: 1.0}
blend_oof  = models[best_name][0].copy().astype(float)
blend_pred = models[best_name][1].copy().astype(float)

TOL = 1e-5  # stop if improvement is less than this

for step in range(20):
    best_step_auc = best_auc
    best_w = None
    best_add = None

    for name, (o, _) in models.items():
        # Try positive weights: new_blend = (blend + w * model) / (1 + w)
        # This normalises so weights always sum to 1
        for w in np.arange(0.05, 1.0, 0.05):
            trial = (blend_oof + w * o) / (1 + w)
            auc = roc_auc_score(y_all, trial)
            if auc > best_step_auc:
                best_step_auc = auc; best_w = w; best_add = name
        # Try negative weights: sometimes subtracting a small amount of a model helps
        # if it's overcorrecting in a particular region
        for w in np.arange(0.05, 0.5, 0.05):
            trial = np.clip((blend_oof - w * o) / (1 - w), 0, 1)
            auc = roc_auc_score(y_all, trial)
            if auc > best_step_auc:
                best_step_auc = auc; best_w = -w; best_add = name

    if best_add is None or (best_step_auc - best_auc) < TOL:
        print(f'Stopped: improvement {best_step_auc - best_auc:.8f} < TOL={TOL}')
        break

    w = best_w
    blend_oof  = np.clip((blend_oof  + w * models[best_add][0]) / (1 + w), 0, 1)
    blend_pred = np.clip((blend_pred + w * models[best_add][1]) / (1 + w), 0, 1)
    weights[best_add] = weights.get(best_add, 0) + w
    best_auc = best_step_auc
    print(f'  Step {step}: AUC {best_auc:.6f} | add {best_add} w={w:.3f}')

print(f'\nFinal ensemble OOF AUC: {roc_auc_score(y_all, blend_oof):.5f}')
print('Weights:', {k: round(v, 3) for k, v in weights.items()})

sub['Churn'] = blend_pred
sub.to_csv('submission_ensemble.csv', index=False)
print('Saved submission_ensemble.csv')

## 9. Pseudo-Labeling

### The idea
We have 254,655 test customers whose true churn label we don't know.
But our ensemble gives us a **probability** for each one.

For customers where the ensemble is very confident — say, 95% probability of churning —
that prediction is probably correct. We can treat these high-confidence predictions as
if they were real labels ('soft pseudo-labels') and add them to our training data.

### Why soft labels?
Instead of rounding to 0 or 1, we keep the probabilities (e.g. 0.92).
This acts as regularisation — we're saying 'this customer probably churned' rather than
'this customer definitely churned'. Using hard labels would overfit to our own predictions.

### The risk
If our ensemble is systematically wrong about certain customers, pseudo-labeling
will reinforce those errors. This is why we only use high-confidence predictions
(threshold 15% — only add customers predicted <15% or >85% churn probability).

In [ ]:
PSEUDO_THRESH = 0.15  # only use predictions below 0.15 or above 0.85
mask = (blend_pred < PSEUDO_THRESH) | (blend_pred > (1 - PSEUDO_THRESH))
print(f'High-confidence test samples: {mask.sum():,} / {len(blend_pred):,} ({mask.mean()*100:.1f}%)')

# Combine original training data with pseudo-labeled test data
X_pseudo = pd.concat([X_all, Xtest_raw[mask].reset_index(drop=True)], ignore_index=True)
y_pseudo  = np.concatenate([y_all.values, blend_pred[mask]])  # soft labels for test portion
print(f'Expanded training set: {len(X_pseudo):,} rows (was {len(X_all):,})')

# Retrain on full data + pseudo-labels (no validation set — we use all data)
# Use fixed n_estimators based on what early stopping found previously
# (can't use early stopping without a held-out val set)
best_iters = model.best_iteration_ if hasattr(model, 'best_iteration_') else 400
n_est = int(best_iters * 1.1)  # slightly more trees since we have more training data
print(f'Retraining LGB with n_estimators={n_est} (no early stopping, using all data)...')

pseudo_params = {**lgb_params, 'n_estimators': n_est, 'random_state': SEED}
pred_pseudo   = np.zeros(len(Xtest_raw), dtype=np.float64)

for seed in LGB_SEEDS:
    Xp = X_pseudo.copy(); Xte = Xtest_raw.copy()
    # Must encode categoricals before fitting
    for c in CATS:
        if c in Xp.columns:
            codes, _ = pd.factorize(pd.concat([Xp[c], Xte[c]], ignore_index=True).astype(str))
            Xp[c]   = codes[:len(Xp)].astype('int32')
            Xte[c]  = codes[len(Xp):].astype('int32')
    for c in Xp.select_dtypes('object').columns:
        codes, _ = pd.factorize(pd.concat([Xp[c], Xte[c]], ignore_index=True).astype(str))
        Xp[c]   = codes[:len(Xp)].astype('int32')
        Xte[c]  = codes[len(Xp):].astype('int32')
    m = lgb.LGBMClassifier(**{**pseudo_params, 'random_state': seed})
    m.fit(Xp, y_pseudo)
    pred_pseudo += m.predict_proba(Xte)[:, 1] / len(LGB_SEEDS)

# Blend: mostly keep the ensemble, add a small contribution from the pseudo-label model
# The weight 0.3 is a hyperparameter — too high risks reinforcing ensemble errors
PSEUDO_WEIGHT = 0.3
final_pred = (1 - PSEUDO_WEIGHT) * blend_pred + PSEUDO_WEIGHT * pred_pseudo
print(f'\nPseudo-label blend: {1-PSEUDO_WEIGHT:.0%} ensemble + {PSEUDO_WEIGHT:.0%} pseudo-model')

sub['Churn'] = final_pred
sub.to_csv('submission_final.csv', index=False)
print('Saved submission_final.csv — this is our best submission')

## Summary

| Technique | Why it helps | Implementation |
|-----------|-------------|----------------|
| Digit features | Exposes pricing clusters to tree splits | Decompose numerics digit-by-digit |
| Round features | Groups similar prices explicitly | Round at multiple scales |
| ORIG stats | Ground-truth churn rates from real data | Map IBM Telco stats onto synthetic data |
| Pairwise interactions | Captures joint effects (contract × internet) | String concat + target encoding |
| OOF target encoding | Prevents data leakage in TE | Encode each row from other folds only |
| OOF predictions | Honest validation score | Each row predicted by model that never saw it |
| Multiple seeds | Reduces random variance | Average 3 LGB models with different seeds |
| CatBoost ensemble | Diverse errors from different algorithm | Level-wise trees + native cat encoding |
| Hill climbing | Optimal model blend weights | Greedy search over weight combinations |
| Pseudo-labeling | More training data from test set | Soft labels for high-confidence test preds |

**Key insight:** The top competitors don't have fundamentally different data or features.
They win by being rigorous about preventing leakage, ensuring diverse models, and
squeezing marginal gains from each technique. Every 0.001 AUC is ~100 rank places.